# Merging of the Bookings and Measurements datasets in respect to the overlapping timeframes in the column created_at

---

In [1]:
import pyarrow.parquet as pq
import duckdb
import os

In [2]:
base_path = r"M:\Universität\Master\Semester2\RealWorld_ML_Problems\Data\data_hella_single_line\filtered"
output_dir = r"M:\Universität\Master\Semester2\RealWorld_ML_Problems\Data\data_hella_single_line\merged"

filtered_measurements_file = os.path.join(base_path, "filtered_measurements_encoded.parquet")
filtered_bookings_file = os.path.join(base_path, "filtered_bookings.parquet")
merged_output_path = os.path.join(output_dir, "merged_bookings_measurements.parquet")

In [3]:
con = duckdb.connect()

### Load files into DuckDB as views (no data copied yet)

In [4]:
con.execute(f"CREATE OR REPLACE VIEW measurements AS SELECT * FROM '{filtered_measurements_file}';")
con.execute(f"CREATE OR REPLACE VIEW bookings AS SELECT * FROM '{filtered_bookings_file}';")

### Calculate overlapping time window, only load created_at column for efficiency

In [5]:
time_window = con.execute(f"""
    SELECT
        GREATEST(
            (SELECT MIN(created_at) FROM '{filtered_measurements_file}'),
            (SELECT MIN(created_at) FROM '{filtered_bookings_file}')
        ) AS start_time,
        LEAST(
            (SELECT MAX(created_at) FROM '{filtered_measurements_file}'),
            (SELECT MAX(created_at) FROM '{filtered_bookings_file}')
        ) AS end_time;
""").fetchdf()

start_time = time_window['start_time'][0]
end_time = time_window['end_time'][0]
print(f"Overlapping Time Window: {start_time} to {end_time}")

Overlapping Time Window: 2025-03-01 02:21:20.773000+01:00 to 2025-05-14 03:02:06.180000+02:00


### Filter datasets by time window and perform the merge

In [6]:
con.execute("""
    CREATE OR REPLACE TABLE merged AS
    SELECT
        m.* EXCLUDE (booking_id),
        b.*
    FROM measurements m
    INNER JOIN bookings b USING (booking_id)
    WHERE m.created_at BETWEEN ? AND ?
      AND b.created_at BETWEEN ? AND ?;
""", [start_time, end_time, start_time, end_time])

### Save merged dataset to Parquet

In [7]:
con.execute(f"COPY merged TO '{merged_output_path}' (FORMAT 'parquet', COMPRESSION 'zstd');")

In [9]:
# Print summary
shape = con.execute("SELECT COUNT(*) AS rows FROM merged;").fetchdf()
print("Merged file saved to:", merged_output_path)
print("Merged shape rows:", shape['rows'][0])
# Print first 10 rows from the saved file
head_df = con.execute(f"SELECT * FROM '{merged_output_path}' LIMIT 10;").df()
print(head_df)

Balanced merged file saved to: M:\Universität\Master\Semester2\RealWorld_ML_Problems\Data\data_hella_single_line\merged\merged_bookings_measurements_balanced.parquet
Balanced merged shape rows: 30000
   measure_step_number  measure_value                       created_at  \
0                    0         4.0000 2025-03-28 04:24:25.486000+01:00   
1                  830       825.1510 2025-05-09 15:10:55.734000+02:00   
2                   72        68.3142 2025-04-22 11:43:20.026000+02:00   
3                   14        10.0000 2025-04-14 02:04:18.572000+02:00   
4                  846       552.0180 2025-04-23 04:31:54.277000+02:00   
5                  664       100.0000 2025-04-22 11:10:10.159000+02:00   
6                  515       100.0020 2025-04-01 09:34:14.804000+02:00   
7                  414        20.2033 2025-04-17 00:09:35.852000+02:00   
8                  506       469.6890 2025-04-23 02:26:46.071000+02:00   
9                  855       836.7920 2025-04-16 17:15:19.38

In [10]:
# Close connection
con.close()